# 09 - Prediction Classification Pix

Modelo educacional para classificar tendência de crescimento do próximo mês.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql import Window
from pyspark.sql import functions as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from src.config import FIGURES_DIR, PIX_CLASSIFICATION_PREDICTIONS_DIR, PIX_ML_FEATURES_DIR, REPORTS_DIR, create_project_directories
from src.data_quality import ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(False)
spark = get_spark_session("09-classification-pix")

In [ ]:
features = ["mes_numero", "trimestre", "valor_total_lag_1", "quantidade_transacoes_lag_1", "ticket_medio_lag_1", "crescimento_valor_lag_1", "crescimento_qtd_lag_1", "valor_total_mm3", "quantidade_transacoes_mm3", "ticket_medio_mm3", "flag_crescimento_valor", "flag_crescimento_qtd"]
w = Window.orderBy("ano_mes")
df = (spark.read.parquet(str(PIX_ML_FEATURES_DIR))
      .withColumn("classe_tendencia_valor_next_month", F.lead("flag_crescimento_valor").over(w))
      .dropna(subset=["classe_tendencia_valor_next_month"]))
for col in features:
    df = df.withColumn(col, F.coalesce(F.col(col).cast("double"), F.lit(0.0)))
df = df.withColumn("target_text", F.when(F.col("classe_tendencia_valor_next_month") == 1, "crescimento").otherwise("nao_crescimento"))
indexer = StringIndexer(inputCol="target_text", outputCol="label", handleInvalid="keep")
indexed = indexer.fit(df).transform(df)
assembler = VectorAssembler(inputCols=features, outputCol="features")
model_df = assembler.transform(indexed).select("ano_mes", "features", "label", "target_text")
rows = model_df.count()
train_count = max(2, int(rows * 0.75))
train_months = [r["ano_mes"] for r in model_df.orderBy("ano_mes").limit(train_count).select("ano_mes").collect()]
train_df = model_df.filter(F.col("ano_mes").isin(train_months))
test_df = model_df.filter(~F.col("ano_mes").isin(train_months))
model = LogisticRegression(featuresCol="features", labelCol="label", predictionCol="prediction", maxIter=20)
fit_model = model.fit(train_df)
predictions = fit_model.transform(test_df).orderBy("ano_mes")
predictions.write.mode("overwrite").parquet(str(PIX_CLASSIFICATION_PREDICTIONS_DIR))
predictions.show(truncate=False)

In [ ]:
metrics = []
for metric in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
    name = {"weightedPrecision": "precision", "weightedRecall": "recall"}.get(metric, metric)
    metrics.append((name, float(evaluator.evaluate(predictions))))
metrics_df = spark.createDataFrame(metrics, ["metric", "value"])
metrics_df.toPandas().to_csv(REPORTS_DIR / "classification_metrics.csv", index=False)
metrics_df.show()

pd_pred = predictions.select("label", "prediction").toPandas()
labels = sorted(set(pd_pred["label"].tolist() + pd_pred["prediction"].tolist()))
matrix = np.zeros((len(labels), len(labels)), dtype=int)
label_to_idx = {label: idx for idx, label in enumerate(labels)}
for _, row in pd_pred.iterrows():
    matrix[label_to_idx[row["label"]], label_to_idx[row["prediction"]]] += 1

plt.figure(figsize=(7, 6))
plt.imshow(matrix, cmap="Blues")
plt.title("Classificação: matriz de confusão")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.xticks(range(len(labels)), [str(int(x)) for x in labels])
plt.yticks(range(len(labels)), [str(int(x)) for x in labels])
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, matrix[i, j], ha="center", va="center", color="black")
plt.colorbar(label="Quantidade")
plt.figtext(0.01, 0.01, "Fonte: dados públicos do Banco Central do Brasil | Modelo educacional", fontsize=9)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(FIGURES_DIR / "07_pix_classification_confusion_matrix.png", dpi=160)
plt.close()

In [ ]:
spark.stop()